---
# Séries temporelles : `conso_meteo`
---

## Packages

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import plotly.graph_objs as go
import seaborn as sns

from scipy.spatial.distance import cdist

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

from keras.callbacks import EarlyStopping
from keras.models import Model
from keras.layers import Add, Concatenate, Dense, Dropout, GlobalAveragePooling1D, Input, LayerNormalization, LSTM, MultiHeadAttention

## 1. Données

### 1.1. Importation

In [ ]:
# En local :
directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
# from google.colab import drive
# drive.mount('/content/drive')
# directory = '/content/drive/MyDrive/Data/'

#### 1.1.1. Consommation électrique et météorologie

In [ ]:
conso_meteo = pd.read_csv(directory + 'conso_meteo.csv',
                          header    = 0,
                          index_col = None,
                          sep       = ',',
                          decimal   = '.')

In [ ]:
conso_meteo.info()

In [ ]:
conso_meteo.head()

In [ ]:
conso_meteo['date'] = pd.to_datetime(conso_meteo['date'])
conso_meteo_jour = conso_meteo.drop(['heure'], axis=1).resample('D', on='date').mean(numeric_only=True)

In [ ]:
conso_meteo_jour.head()

#### 1.1.2. Jours fériés

In [ ]:
jours_feries = pd.read_csv(directory + 'jours_feries_USA.csv',
                           header    = 0,
                           index_col = None,
                           sep       = ',',
                           decimal   = '.')

In [ ]:
jours_feries.info()

In [ ]:
jours_feries.head()

In [ ]:
jours_feries['date'] = pd.to_datetime(jours_feries['date'], dayfirst=True)
jours_feries['jf']   = 1

In [ ]:
jours_feries.head()

### 1.2. Concaténation

In [ ]:
data = conso_meteo.merge(jours_feries, on='date', how='left')
data['jf'] = data['jf'].fillna(0).astype(int)

In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data_jour = conso_meteo_jour.merge(jours_feries, on='date', how='left')
data_jour['jf'] = data_jour['jf'].fillna(0).astype(int)

In [ ]:
data_jour.head()

In [ ]:
data.info()

### 1.3. Gestion des données manquantes

In [ ]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

### 1.4. Intégration des informations calendaires

In [ ]:
data['datehm'] = pd.to_datetime(data['date']) + pd.to_timedelta(data['heure'], unit='h')

In [ ]:
data['jour_sem']    = data['date'].dt.day_of_week # Le lundi est 0, le dimanche est 6
data['we']          = np.where(data['jour_sem'].isin([5, 6]), 1, 0)
data['an']          = data['date'].dt.year
data['num_jour_an'] = data['date'].dt.dayofyear

In [ ]:
data_jour['jour_sem']    = data_jour['date'].dt.day_of_week
data_jour['we']          = np.where(data_jour['jour_sem'].isin([5, 6]), 1, 0)
data_jour['an']          = data_jour['date'].dt.year
data_jour['num_jour_an'] = data_jour['date'].dt.dayofyear

In [ ]:
first_datehm = data['datehm'].iloc[0]
last_datehm  = data['datehm'].iloc[-1]

print("Premier instant :", first_datehm)
print("Dernier instant :", last_datehm)

## 2. Analyse descriptive succincte des données

### 2.1. Séries temporelles

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=data['datehm'], y=data['LOAD'], mode='lines', name='Load'))

fig.update_layout(
    title='Analyse temporelle de la consommation électrique',
    xaxis_title='Temps',
    yaxis_title='Consommation',
)

fig.update_layout(
    hovermode='x unified',
    xaxis=dict(rangeselector=dict(
        buttons=list([
            dict(count=1, label='1m', step='month', stepmode='backward'),
            dict(count=6, label='6m', step='month', stepmode='backward'),
            dict(step='all')
        ])
    ),
    rangeslider=dict(visible=True),
    type='date')
)

fig.show()

In [ ]:
fig = go.Figure()

for i in range(1, 26):
    col = f'w{i}'
    fig.add_trace(go.Scatter(
        x=data['datehm'],
        y=data[col],
        mode='lines',
        name=col
    ))

fig.update_layout(
    title='Analyse des températures des stations w1-w25',
    xaxis_title='Temps',
    yaxis_title='Température',
    hovermode='x unified',
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label='1m', step='month', stepmode='backward'),
                dict(count=6, label='6m', step='month', stepmode='backward'),
                dict(step='all')
            ])
        ),
        rangeslider=dict(visible=True),
        type='date'
    )
)

fig.show()

In [ ]:
plt.figure(figsize=(14, 6))
for an, group in data.groupby('an'):
    plt.plot(group['num_jour_an'], group['LOAD'], label=str(an), alpha=0.8)

plt.title('Consommation électrique par année')
plt.xlabel("Jour de l'année")
plt.ylabel('Consommation')
plt.legend(title='Année', loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
for an, group in data_jour.groupby('an'):
    plt.plot(group['num_jour_an'], group['LOAD'], label=str(an), alpha=0.8)

plt.title('Consommation électrique par année (moyenne journalière)')
plt.xlabel("Jour de l'année")
plt.ylabel('Consommation')
plt.legend(title='Année', loc='upper right')
plt.grid(True)
plt.tight_layout()
plt.show()

### 2.2. Nuages de points

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(data['w1'], data['LOAD'], alpha=0.3, s=10)
plt.title('Nuage de points')
plt.xlabel('Température station 1')
plt.ylabel('Consommation électrique')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(data['w1'], data['LOAD'], c=data['heure'], cmap='viridis', alpha=0.5, s=10)

cbar = plt.colorbar(scatter)
cbar.set_label('Heure')

plt.title('Nuage de points')
plt.xlabel('Température station 1')
plt.ylabel('Consommation électrique')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
data_9 = data.loc[data['heure']==9,]

In [ ]:
cmap = plt.get_cmap('coolwarm')

plt.figure(figsize=(8, 6))
scatter = plt.scatter(data_9['w1'], data_9['LOAD'], c=data_9['we'], cmap=cmap, alpha=0.5, s=10)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Semaine',
           markerfacecolor=cmap(0.0), markersize=8, alpha=0.5),
    Line2D([0], [0], marker='o', color='w', label='Week-end',
           markerfacecolor=cmap(1.0), markersize=8, alpha=0.5)
]

plt.legend(handles=legend_elements, title='Jour', loc='upper right')
plt.title('Nuage de points (9h)')
plt.xlabel('Température station 1')
plt.ylabel('Consommation électrique')
plt.grid(True)
plt.tight_layout()
plt.show()

## 3. Sélection des variables météorologiques par clustering

In [ ]:
data_cluster = data[[f'w{i}' for i in range(1, 26)]].transpose()

In [ ]:
data_cluster.columns = [f't{i}' for i in range(data_cluster.shape[1])]

In [ ]:
k_max = 10

inertie_intra = pd.DataFrame(columns=['k', 'inertie_intra'])

for k in range(1, k_max+1):
    kmeans_model = KMeans(init='k-means++', max_iter=100, n_clusters=k, n_init=5)
    kmeans_out = kmeans_model.fit(data_cluster)
    inertie_intra.loc[k-1,'k'] = k
    inertie_intra.loc[k-1,'inertie_intra'] = kmeans_out.inertia_ * 100

inertie_intra['part_inertie_intra'] = 100 * inertie_intra['inertie_intra'] / inertie_intra['inertie_intra'][0]

In [ ]:
sns.set_theme(style='darkgrid')
fig, ax = plt.subplots(figsize=(8, 6))
ax.vlines(inertie_intra['k'], 0, inertie_intra['part_inertie_intra'], linewidth=10)
ax.set_xticks(range(1, k_max+1))
ax.set_xlabel('K')
ax.set_ylabel("Part d'inertie intra-classes (%)")
plt.title('K-means')
plt.show()

In [ ]:
k_choice = 3

In [ ]:
kmeans_model = KMeans(init='k-means++', max_iter=300, n_clusters=k_choice, n_init=2)
kmeans_out = kmeans_model.fit(data_cluster)

data_cluster['cluster'] = kmeans_out.fit_predict(data_cluster)
data_cluster['cluster'].value_counts()

In [ ]:
for i in range(k_choice):
    print('\n', 'Cluster', i)
    print(list(data_cluster[data_cluster['cluster'] == i].index))

In [ ]:
features = [col for col in data_cluster.columns if col not in ['cluster']]

On passe d'un array 1D (n_features,) à un array 2D (1, n_features) pour utiliser la commande `cdist`.

In [ ]:
for i in range(kmeans_out.n_clusters):
    cluster_i = data_cluster[data_cluster['cluster'] == i]
    X_i = cluster_i[features].values
    centroid_i = kmeans_out.cluster_centers_[i].reshape(1, -1)
    distances_i = cdist(X_i, centroid_i).flatten()
    data_cluster.loc[cluster_i.index, 'dist_centroid'] = distances_i

data_cluster[['cluster', 'dist_centroid']].sort_values(by=['cluster', 'dist_centroid'], ascending=[True, True])

On conserve ici une donnée météorologique par cluster (la plusproche du centroïde) : w3, w7 et w24. Attention cet interclassement peut différer à la marge suivant l'initialisation de l'algorithme des K-means.

In [ ]:
cols_to_keep = ['w3', 'w7', 'w24']
cols_to_drop = [col for col in data.columns if col.startswith('w') and col not in cols_to_keep]
data_model = data.copy().drop(columns=cols_to_drop)

On considère dans la suite que les prévisions météorologiques seront disponibles et on évaluera les modèles à "conditions clmimatiques réalisées".

## 4. Prévision par machine learning "classique"

On considère 2 options de feature engineering temporel :
1. Intégration de variables retard
2. Intégration de fonctions temporelles (déterministes)

Quelle que soit l'option, on effectue un *one hot encoding* du type de jour :

In [ ]:
data_model = pd.get_dummies(data_model, columns=['jour_sem'], drop_first=True)

In [ ]:
cols_dummies = [col for col in data_model.columns if col.startswith('jour_sem')]
data_model[cols_dummies] = data_model[cols_dummies].astype(int)

In [ ]:
data_model.info()

### 4.1. Intégration de variables retard

#### 4.1.1. Modèle par heure et horizon de prévision

On considère le modèle suivant :
$$
L_{t+\ell}=f_{h}\left(L_{t}, \ldots, L_{t-3}, W_{t+\ell}^{(3)}, W_{t+\ell}^{(7)}, W_{t+\ell}^{(24)},F_{t+\ell}, S_{t+\ell}^{(Ma)}, \ldots, S_{t+\ell}^{(Di)}\right)+\varepsilon_{t+\ell}
$$
où $\ell\in\left\{1,\ldots,24\right\}$ désigne l'horizon de prévision, $h$ l'heure, $F$ la covariable "jour férié" et $S^{\text{(jour semaine)}}$ les covariables "jour de semaine" (6 indicatrices).

On procède à l'aide d'un random forest ici.

In [ ]:
# Prévision jusqu'à l'horizon 24
horizon = 24

# Date de séparation des échantillons
split_date = pd.Timestamp('2010-01-01 00:00:00')

# Création des dictionnaires pour stocker les modèles et les critères d'évaluation
modeles1a = {}
rmse_heure1a = {}
mape_heure1a = {}
rmse1a = {}
mape1a = {}

for hor in range(horizon):
    # Création des variables retard
    lags = [1, 2, 3, 4]
    data_model_t = data_model.copy()
    
    for lag in lags:
        data_model_t[f'LOAD_t-{lag}'] = data_model_t['LOAD'].shift(lag)
    
    # Décalage temporel des variables LOAD, w3, w7 et w24 à l'horizon de prévision
    data_model_t['LOAD'] = data_model_t['LOAD'].shift(-hor)
    data_model_t['w3']   = data_model_t['w3'].shift(-hor)
    data_model_t['w7']   = data_model_t['w7'].shift(-hor)
    data_model_t['w24']  = data_model_t['w24'].shift(-hor)

    data_model_t.dropna(inplace=True)
    
    # Séparation train / test
    data_train = data_model_t[data_model_t['datehm'] < split_date].copy()
    data_test = data_model_t[data_model_t['datehm'] >= split_date].copy()

    # Boucle sur les 24 heures de la journée
    for heur in range(24):

        # Filtrage sur l'heure
        data_train_heure = data_train[data_train['heure'] == heur]
        data_test_heure  = data_test[data_test['heure'] == heur]
        
        # Covariables
        cols_to_drop = ['heure', 'date', 'datehm', 'an', 'num_jour_an']
        features = [col for col in data_train.columns if col not in cols_to_drop]
        target = 'LOAD'

        # Séparation échantillons d'apprentissage et de test
        X_train = data_train_heure[features]
        y_train = data_train_heure[target]
        
        X_test = data_test_heure[features]
        y_test = data_test_heure[target]
        
        # Random Forest : estimation
        rf = RandomForestRegressor(n_estimators=100)
        rf.fit(X_train, y_train)
        
        # Random Forest : prévision et critères d'erreur
        y_pred = rf.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        mape = mean_absolute_percentage_error(y_test, y_pred)
        
        # Stockage du modèle et des critères d'évaluation
        modeles1a[(hor, heur)] = rf
        rmse_heure1a[(hor, heur)] = rmse
        mape_heure1a[(hor, heur)] = mape * 100

In [ ]:
mape1a_0h = [mape_heure1a[(hor, 0)] for hor in range(24)]

plt.figure(figsize=(8, 5))
plt.plot(range(1, 25), mape1a_0h, marker='o')
plt.xlabel('Horizon de prévision')
plt.ylabel('MAPE (%)')
plt.title('Modèle 1 : h=0')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mape1a_hor24 = [mape_heure1a[(23, heur)] for heur in range(24)]

plt.figure(figsize=(8, 5))
plt.plot(range(1, 25), mape1a_hor24, marker='o')
plt.xlabel('Heure')
plt.ylabel('MAPE (%)')
plt.title('Modèle 1 : $\ell=24$')
plt.grid(True)
plt.tight_layout()
plt.show()

#### 4.1.2. Modèle par horion de prévision

On considère le modèle suivant :
$$
L_{t+\ell}=f\left(L_{t}, \ldots, L_{t-3}, W_{t+\ell}^{(3)}, W_{t+\ell}^{(7)}, W_{t+\ell}^{(24)},F_{t+\ell}, S_{t+\ell}^{(Ma)}, \ldots, S_{t+\ell}^{(Di)}\right)+\varepsilon_{t+\ell}
$$
où $\ell\in\left\{1,\ldots,24\right\}$ désigne l'horizon de prévision, $F$ la covariable "jour férié" et $S^{\text{(jour semaine)}}$ les covariables "jour de semaine" (6 indicatrices).

On procède à l'aide d'un random forest ici.

In [ ]:
# Prévision jusqu'à l'horizon 24
horizon = 24

# Date de séparation des échantillons
split_date = pd.Timestamp('2010-01-01 00:00:00')

# Création des dictionnaires pour stocker les modèles et les critères d'évaluation
modeles1b = {}
rmse1b = {}
mape1b = {}

for hor in range(horizon):
    # Création des variables retard
    lags = [1, 2, 3, 4]
    data_model_t = data_model.copy()
    
    for lag in lags:
        data_model_t[f'LOAD_t-{lag}'] = data_model_t['LOAD'].shift(lag)
    
    # Décalage temporel des variables LOAD, w3, w7 et w24 à l'horizon de prévision
    data_model_t['LOAD'] = data_model_t['LOAD'].shift(-hor)
    data_model_t['w3']   = data_model_t['w3'].shift(-hor)
    data_model_t['w7']   = data_model_t['w7'].shift(-hor)
    data_model_t['w24']  = data_model_t['w24'].shift(-hor)

    data_model_t.dropna(inplace=True)
    
    # Séparation train / test
    data_train = data_model_t[data_model_t['datehm'] < split_date].copy()
    data_test = data_model_t[data_model_t['datehm'] >= split_date].copy()

    # Covariables
    cols_to_drop = ['heure', 'date', 'datehm', 'an', 'num_jour_an']
    features = [col for col in data_train.columns if col not in cols_to_drop]
    target = 'LOAD'

    # Séparation échantillons d'apprentissage et de test
    X_train = data_train[features]
    y_train = data_train[target]
        
    X_test = data_test[features]
    y_test = data_test[target]
        
    # Random Forest : estimation
    rf = RandomForestRegressor(n_estimators=100)
    rf.fit(X_train, y_train)
        
    # Random Forest : prévision et critères d'erreur
    y_pred = rf.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
        
    # Stockage du modèle et des critères d'évaluation
    modeles1b[hor] = rf
    rmse1b[hor] = rmse
    mape1b[hor] = mape * 100

In [ ]:
mape1b = [mape1b[hor] for hor in range(24)]

plt.figure(figsize=(8, 5))
plt.plot(range(1, 25), mape1b, marker='o')
plt.xlabel('Horizon de prévision')
plt.ylabel('MAPE (%)')
plt.title('Modèle 1 : h=0')
plt.grid(True)
plt.tight_layout()
plt.show()

### 4.3. Intégration de variables temporelles (déterministes)

On considère le modèle suivant :
$$
L_t=f\left(C_t^{(h)}, S_t^{(h)}, C_t^{(j,1)}, S_t^{(j,1)}, C_t^{(j,2)}, S_t^{(j,2)},F_t, S_t^{(Ma)}, \ldots, S_t^{(Di)}, W_t^{(3)}, W_t^{(7)}, W_t^{(24)}\right)+\varepsilon_t
$$
où $C_t^{(h)}=\cos\left(\frac{2\pi h}{24}\right)$, $S_t^{(h)}=\sin\left(\frac{2\pi h}{24}\right)$, $C_t^{(j,k)}=\cos\left(\frac{2\pi jk}{365.25}\right)$, $S_t^{(j,k)}=\sin\left(\frac{2\pi jk}{365.25}\right)$, $k\in\mathbb{N}^\star$, $h$ désigne l'heure, $j$ la position du jour dans l'année, $F$ la covariable "jour férié" et $S^{\text{(jour semaine)}}$ les covariables "jour de semaine" (6 indicatrices).

On procède à l'aide d'un random forest ici.

In [ ]:
# Date de séparation des échantillons
split_date = pd.Timestamp('2010-01-01 00:00:00')

# Création des dictionnaires pour stocker les modèles et les critères d'évaluation
modeles2 = {}
rmse2 = {}
mape2 = {}

data_model_t = data_model.copy()
data_model_t['heure_sin'] = np.sin(2 * np.pi * data_model_t['heure'] / 24)
data_model_t['heure_cos'] = np.cos(2 * np.pi * data_model_t['heure'] / 24)

data_model_t['jour_cos1'] = np.cos(2 * np.pi * data_model_t['num_jour_an'] / 365.25)
data_model_t['jour_sin1'] = np.sin(2 * np.pi * data_model_t['num_jour_an'] / 365.25)

data_model_t['jour_cos2'] = np.cos(2 * np.pi * data_model_t['num_jour_an'] * 2 / 365.25)
data_model_t['jour_sin2'] = np.sin(2 * np.pi * data_model_t['num_jour_an'] * 2 / 365.25)

# Séparation train / test
data_train = data_model_t[data_model_t['datehm'] < split_date].copy()
data_test = data_model_t[data_model_t['datehm'] >= split_date].copy()
    
# Covariables
cols_to_drop = ['heure', 'date', 'datehm', 'an', 'num_jour_an']
features = [col for col in data_train.columns if col not in cols_to_drop]
target = 'LOAD'
# Séparation échantillons d'apprentissage et de test
X_train = data_train[features]
y_train = data_train[target]
    
X_test = data_test[features]
y_test = data_test[target]
    
# Random Forest : estimation
rf = RandomForestRegressor(n_estimators=100)
rf.fit(X_train, y_train)
    
# Random Forest : prévision et critères d'erreur
y_pred = rf.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
    
# Stockage du modèle et des critères d'évaluation
modeles2 = rf
rmse2 = rmse
mape2 = mape * 100

In [ ]:
print(f"MAPE : {mape2:.2f}%")

## 5. Prévision à l'aide du deep learning

### 5.1. Préparation des données

On distingue les covariables (le passé de la série et les informations calendaires-météorologiques disponibles dans le futur) :

In [ ]:
past_features = ['LOAD']
future_features = ['jf', 'jour_sem_1', 'jour_sem_2', 'jour_sem_3', 'jour_sem_4', 'jour_sem_5', 'jour_sem_6', 'w24', 'w3', 'w7']
target = 'LOAD'

On considère les 24 derniers instants dans le modèle et on prévoit jusqu'à un horizon de 24 heures :

In [ ]:
n_input  = 24
n_output = 24

On distingue les données d'apprentissage et de test :

In [ ]:
split_date = pd.Timestamp('2010-01-01 00:00:00')

data_train = data_model_t[data_model_t['datehm'] < split_date].copy()
data_test  = data_model_t[data_model_t['datehm'] >= split_date].copy()

On normalise les données sur les données d'apprentissage (à l'aide de `fit_transform`) et on l'applique aux données d'apprentissage et de test (`transform`) :

In [ ]:
scaler_past   = StandardScaler()
scaler_future = StandardScaler()
scaler_y      = StandardScaler()

past_train   = scaler_past.fit_transform(data_train[past_features])
future_train = scaler_future.fit_transform(data_train[future_features])
target_train = scaler_y.fit_transform(data_train[[target]])

past_test   = scaler_past.transform(data_test[past_features])
future_test = scaler_future.transform(data_test[future_features])
target_test = scaler_y.transform(data_test[[target]])

On prépare les séquences :

In [ ]:
def prepare_sequences(past, future, y, n_input, n_output):
    Xp, Xf, yt = [], [], []
    for i in range(len(past) - n_input - n_output + 1):
        Xp.append(past[i:i+n_input])
        Xf.append(future[i+n_input:i+n_input+n_output])
        yt.append(y[i+n_input:i+n_input+n_output].flatten())
    return np.array(Xp), np.array(Xf), np.array(yt)

X_past_train, X_future_train, y_train = prepare_sequences(past_train, future_train, target_train, n_input, n_output)
X_past_test, X_future_test, y_test = prepare_sequences(past_test, future_test, target_test, n_input, n_output)

### 5.2. LSTM

#### 5.2.1. Modèle global

On définit le LSTM (de manière "fonctionnelle") :

In [ ]:
input_past   = Input(shape=(n_input, X_past_train.shape[2]), name='input_past')
input_future = Input(shape=(n_output, X_future_train.shape[2]), name='input_future')

x_past   = LSTM(64, return_sequences=False)(input_past)
x_future = LSTM(32, return_sequences=False)(input_future)
x = Concatenate()([x_past, x_future])
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(n_output)(x)

model = Model(inputs=[input_past, input_future], outputs=output)
model.compile(optimizer='adam', loss='mse')
model.summary()

On estime le modèle :

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    [X_past_train, X_future_train],
    y_train,
    validation_split = 0.1,
    epochs           = 50,
    batch_size       = 64,
    callbacks        = [early_stop],
    verbose          = 1
)

On évalue l'erreur sur l'échantillon test :

In [ ]:
y_pred_scaled = model.predict([X_past_test, X_future_test])

y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

mape = mean_absolute_percentage_error(y_true.flatten(), y_pred.flatten()) * 100
print(f"MAPE : {mape:.2f}%")

In [ ]:
i = 20

plt.figure(figsize=(10, 4))
plt.plot(range(n_output), y_true[i], label='Valeur observée')
plt.plot(range(n_output), y_pred[i], label='Prévision')
plt.title('Prévision de LOAD sur 24 heures')
plt.xlabel('Horizon (heures)')
plt.ylabel('LOAD')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### 5.2.2. Modèle par horizon de prévision

In [ ]:
def prepare_single_horizon(past, future, y, n_input=24, horizon=5):
    Xp, Xf, yt = [], [], []
    for i in range(len(past) - n_input - horizon + 1):
        Xp.append(past[i:i+n_input])
        Xf.append(future[i+n_input:i+n_input+horizon][-1])  # à t+h
        yt.append(y[i+n_input + horizon - 1][0])  # à t+h
    return np.array(Xp), np.array(Xf), np.array(yt)

In [ ]:
# Prévision jusqu'à l'horizon 24
horizon = 24

models = []

for h in range(1, horizon + 1):
    print(f"Horizon {h}")
    Xp_train, Xf_train, y_train_h = prepare_single_horizon(past_train, future_train, target_train, n_input=24, horizon=h)
    Xp_test, Xf_test, y_test_h = prepare_single_horizon(past_test, future_test, target_test, n_input=24, horizon=h)

    input_past = Input(shape=(Xp_train.shape[1], Xp_train.shape[2]))
    input_future = Input(shape=(Xf_train.shape[1],))
    x1 = LSTM(64)(input_past)
    x2 = Dense(32, activation='relu')(input_future)
    x = Concatenate()([x1, x2])
    x = Dense(64, activation='relu')(x)
    out = Dense(1)(x)
    model = Model(inputs=[input_past, input_future], outputs=out)
    model.compile(optimizer='adam', loss='mse')

    model.fit([Xp_train, Xf_train], y_train_h, validation_split=0.1, epochs=20, batch_size=64, verbose=0)
    models.append(model)

In [ ]:
y_preds = []

for h, model in enumerate(models):
    Xp_test_h, Xf_test_h, y_test_h = prepare_single_horizon(past_test, future_test, target_test, n_input=24, horizon=h+1)
    y_pred_h = model.predict([Xp_test_h, Xf_test_h])
    y_preds.append(y_pred_h.flatten())

In [ ]:
for h in range(horizon):
    y_true_h = scaler_y.inverse_transform(target_test[n_input + h: n_input + h + len(y_preds[h])])
    y_pred_h = scaler_y.inverse_transform(y_preds[h].reshape(-1, 1))
    mape_h = mean_absolute_percentage_error(y_true_h, y_pred_h)
    print(f"Horizon t+{h+1} : MAPE = {mape_h*100:.2f}%")